# 🏆 نظام تصنيف لاعبي الفانتازي باستخدام الشبكات العصبية (ANN)
## المرحلة 1: تحميل البيانات وتجهيزها (Data Loading & Preprocessing)
---
في هذه المرحلة سنقوم بتحميل بيانات الدوري الإنجليزي 2025/26 وتنظيفها وإنشاء فئات التصنيف المطلوبة.

In [ ]:
# 1. تثبيت واستيراد المكتبات
!pip install kagglehub tensorflow scikit-learn pandas numpy matplotlib seaborn

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [ ]:
# 2. تحميل البيانات من Kaggle
path = kagglehub.dataset_download('calvinrostanto/fantasy-premier-league-2025-2026')
print('✅ Path to dataset files:', path)

# البحث عن ملفات الـ CSV
csv_files = glob.glob(os.path.join(path, '**', 'players.csv'), recursive=True)
if not csv_files:
    csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)

df_raw = pd.read_csv(csv_files[0])
print(f'📊 تم تحميل البيانات بنجاح. الأبعاد: {df_raw.shape}')
df_raw.head()

## 🧹 تنظيف البيانات (Cleaning)
سنقوم بإزالة القيم الفارغة، التكرارات، والأعمدة غير الضرورية.

In [ ]:
df = df_raw.copy()

# 1. إزالة التكرارات
df = df.drop_duplicates()

# 2. اختيار الأعمدة الهامة للتحليل
cols_to_keep = [
    'total_points', 'minutes', 'goals_scored', 'assists', 
    'clean_sheets', 'goals_conceded', 'saves', 'bonus', 'bps', 
    'influence', 'creativity', 'threat', 'ict_index', 'now_cost', 
    'form', 'selected_by_percent', 'element_type'
]
df = df[cols_to_keep]

# 3. معالجة القيم المفقودة
df = df.fillna(0)

print(f'✅ البيانات بعد التنظيف: {df.shape}')

## 🛠️ هندسة الميزات وإنشاء الأهداف (Target Engineering)
سنقوم الآن بإنشاء الثلاث تصنيفات المطلوبة:

In [ ]:
# 1. هدف فئة النقاط (Points Tier)
def classify_points(pts):
    if pts >= 8: return 'Star'
    elif pts >= 2: return 'Standard'
    else: return 'Blank'

df['points_tier'] = df['total_points'].apply(classify_points)

# 2. هدف قيمة السعر (Value for Money)
df['value_ratio'] = df['total_points'] / (df['now_cost'] + 1)
q1, q2 = df['value_ratio'].quantile([0.33, 0.66])

def classify_value(val):
    if val <= q1: return 'Overpriced'
    elif val <= q2: return 'Fairly priced'
    else: return 'Underpriced'

df['value_tier'] = df['value_ratio'].apply(classify_value)

# 3. هدف الكلين شيت (Clean Sheet Potential)
def classify_cs(cs):
    return 'High CS Chance' if cs >= 5 else 'Low CS Chance'

df['cs_potential'] = df['clean_sheets'].apply(classify_cs)

print('✅ تم إنشاء الأعمدة المستهدفة (Targets) بنجاح.')

## ⚙️ المرحلة 2: التشفير وتقسيم البيانات (Encoding & Splitting)
في هذه المرحلة سنقوم بتحويل البيانات النصية إلى أرقام، وعمل Scaling للميزات، وتقسيم البيانات لتدريب الموديل.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

# 1. تحديد الميزات (Features) والهدف (Target)
X = df.drop(columns=['points_tier', 'value_tier', 'cs_potential', 'total_points', 'value_ratio'])
y = df['points_tier']

# 2. تشفير الهدف (Label Encoding)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

# 3. تقسيم البيانات (70% تدريب، 15% تحقق، 15% اختبار)
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# 4. عمل Scaling للميزات
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 5. تحويل الهدف إلى One-Hot Encoding
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print(f'✅ تم تجهيز البيانات.')
print(f'Train size: {X_train_scaled.shape}, Classes: {le.classes_}')

## 🧠 المرحلة 3: بناء نموذج الشبكة العصبية (Building ANN)
سنقوم ببناء شبكة عصبية مكونة من عدة طبقات مخفية مع استخدام `ReLU` كدالة تنشيط للطبقات المخفية و `Softmax` للطبقة الأخيرة.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

def build_model(input_dim, num_classes):
    model = Sequential([
        # الطبقة الأولى
        Dense(128, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.2),
        
        # الطبقة الثانية
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        
        # الطبقة الثالثة
        Dense(32, activation='relu'),
        
        # الطبقة الأخيرة (Output Layer)
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(X_train_scaled.shape[1], num_classes)
model.summary()

## 🏋️ المرحلة 4: التدريب والتقييم (Training & Evaluation)
سنقوم بتدريب الموديل واستخدام `EarlyStopping` لمنع الـ Overfitting.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

print('✅ انتهى التدريب.')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. التقييم على مجموعة الاختبار
loss, acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f'🎯 Accuracy on Test Set: {acc*100:.2f}%')

# 2. تقرير التصنيف الشامل
y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 3. رسم مصفوفة الارتباك (Confusion Matrix)
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.show()

## 🔮 المرحلة 5: التنبؤ على بيانات جديدة (Prediction)
سنقوم ببناء دالة تأخذ إحصائيات لاعب جديد وتعطينا الفئة المتوقعة.

In [ ]:
def predict_player_tier(stats_dict):
    # تحويل الـ dict لـ DataFrame بنفس ترتيب الـ Features
    input_df = pd.DataFrame([stats_dict])
    
    # التأكد من وجود كل الأعمدة
    for col in X.columns:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[X.columns]
    
    # عمل Scaling بنفس الـ scaler الأصلي
    input_scaled = scaler.transform(input_df)
    
    # التنبؤ
    pred_probs = model.predict(input_scaled, verbose=0)
    pred_class = le.classes_[np.argmax(pred_probs)]
    confidence = np.max(pred_probs) * 100
    
    return pred_class, confidence

# --- تجربة لاعب جديد (مثال: محمد صلاح) ---
new_player_stats = {
    'minutes': 2800,
    'goals_scored': 20,
    'assists': 10,
    'clean_sheets': 1,
    'goals_conceded': 30,
    'saves': 0,
    'bonus': 25,
    'bps': 600,
    'influence': 900.0,
    'creativity': 700.0,
    'threat': 1000.0,
    'ict_index': 260.0,
    'now_cost': 125,
    'form': 8.5,
    'selected_by_percent': 35.0,
    'element_type': 3 # MID
}

tier, conf = predict_player_tier(new_player_stats)
print(f'🏅 اللاعب ينتمي لفئة: {tier} (بثقة {conf:.2f}%)')